# Strategic Analysis: ROI of Crisis-Response Port Orchestration

**APM Terminals Rotterdam MVII | Q1 2026 | Synthetic Data Model**

This notebook documents the quantitative basis for the $19.23M ROI figure reported in the project README. It covers data validation, crisis period characterisation, scenario comparison, and the strategic implication that follows from the findings.

All figures derive from synthetically generated data calibrated to public Rotterdam port statistics. No proprietary Maersk data was used.

## 1.0 Data Scope & Assumption Modeling

Ingesting transaction-level digital twin data across 563 vessel records covering Q1 2026. Terminal capacity baseline is 2.7M TEU (Rotterdam MVII current). A non-linear efficiency cliff penalty is assumed above 80% yard utilization — consistent with published terminal congestion research.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Maersk color palette — consistent with dashboard
plt.rcParams['figure.facecolor'] = '#002147'
plt.rcParams['axes.facecolor']   = '#00233D'
plt.rcParams['text.color']       = '#EDEDED'
plt.rcParams['axes.labelcolor']  = '#878787'
plt.rcParams['xtick.color']      = '#878787'
plt.rcParams['ytick.color']      = '#878787'
plt.rcParams['axes.spines.right']  = False
plt.rcParams['axes.spines.top']    = False

# Load datasets
vessels   = pd.read_csv('../01-data/synthetic/rwg_digital_twin_q1.csv')
scenarios = pd.read_csv('../01-data/synthetic/tableau_scenarios.csv')
heatmap   = pd.read_csv('../01-data/synthetic/heatmap_data.csv')
weekly    = pd.read_csv('../01-data/synthetic/stacked_bar_data.csv')

print(f'Vessel records ingested:    {len(vessels)}')
print(f'Date range:                 {vessels["Date"].min()}  →  {vessels["Date"].max()}')
print(f'Routes:                     {sorted(vessels["Route"].unique())}')
print(f'Scenario models available:  {scenarios["Scenario"].tolist()}')

## 2.0 Operational Integrity Audit

Validating crisis window parameters (Weeks 5–9) and confirming that route distributions reflect the documented 66% Suez diversion pattern. The key diagnostic is the contrast between crisis-week and non-crisis-week wait times — this delta is the operational case for intervention.

In [ ]:
print('=== COMPLETENESS CHECK ===')
print(f'Missing values:\n{vessels.isnull().sum()}\n')

print('=== ROUTE DISTRIBUTION ===')
route_counts = vessels['Route'].value_counts()
for route, count in route_counts.items():
    print(f'  {route:<8} {count} vessels  ({count/len(vessels):.0%})')

print('\n=== CRISIS WINDOW CONTRAST ===')
crisis     = heatmap[heatmap['Is_Crisis_Week'] == 1]
non_crisis = heatmap[heatmap['Is_Crisis_Week'] == 0]

crisis_avg     = crisis['Avg_Wait'].mean()
non_crisis_avg = non_crisis['Avg_Wait'].mean()
surge_multiplier = crisis_avg / non_crisis_avg if non_crisis_avg > 0 else float('inf')

print(f'  Crisis weeks avg wait:     {crisis_avg:.1f}h')
print(f'  Non-crisis weeks avg wait: {non_crisis_avg:.1f}h')
print(f'  Surge multiplier:          {surge_multiplier:.1f}x')
print(f'  Crisis weeks (W5-W9):      {sorted(crisis["Week_Num"].unique())}')
print(f'\n  Yard utilization range:    '
      f'{vessels["Yard_Util_Pct"].min():.1%}  –  {vessels["Yard_Util_Pct"].max():.1%}')
print(f'  Crane MPH (avg):           {vessels["Crane_MPH"].mean():.0f} moves/hour')

## 3.0 Crisis Window Characterisation

The February surge (Weeks 5–9) is not random variance — it is the structural consequence of Cape-rerouted vessels arriving simultaneously with residual Suez traffic. The chart below confirms the pattern and establishes the 16-hour SLA threshold as the relevant operational boundary.

In [ ]:
weekly_agg = heatmap.groupby('Week_Num').agg(
    avg_wait =('Avg_Wait',      'mean'),
    is_crisis=('Is_Crisis_Week', 'first')
).reset_index()

bar_colors = ['#D62828' if c else '#42B0D5' for c in weekly_agg['is_crisis']]

fig, ax = plt.subplots(figsize=(12, 4))

ax.bar(weekly_agg['Week_Num'], weekly_agg['avg_wait'],
       color=bar_colors, edgecolor='none', width=0.7)
ax.axhline(y=16, color='#FFD028', linestyle='--',
           linewidth=1.3, label='SLA threshold — 16h')

ax.set_title('Average Berth Wait Time by Week — Q1 2026',
             color='#EDEDED', fontsize=12, fontweight='bold', pad=14)
ax.set_xlabel('Week Number', color='#878787', fontsize=9)
ax.set_ylabel('Avg Wait (hours)', color='#878787', fontsize=9)
ax.set_xticks(weekly_agg['Week_Num'])

for spine in ax.spines.values():
    spine.set_edgecolor('#1a4a6a')

crisis_p = mpatches.Patch(color='#D62828', label='Crisis weeks (W5–W9)')
normal_p = mpatches.Patch(color='#42B0D5', label='Normal weeks')
thresh_l = plt.Line2D([0], [0], color='#FFD028',
                      linestyle='--', label='SLA threshold (16h)')
ax.legend(handles=[crisis_p, normal_p, thresh_l],
          facecolor='#001628', labelcolor='#EDEDED',
          fontsize=9, edgecolor='#1a4a6a')

plt.tight_layout()
plt.savefig('crisis_window_profile.png', dpi=150,
            bbox_inches='tight', facecolor='#002147')
plt.show()

## 4.0 Comparative ROI Analysis

Quantifying the delta between the unmanaged Crisis Baseline and the Hybrid Optimized orchestration strategy. ROI is read directly from the scenario model's `ROI_M` column, which nets intervention costs against gross savings. The breakdown into SLA and idle cost components is shown separately to identify which cost pool each intervention lever addresses most effectively.

In [ ]:
baseline = scenarios[scenarios['Scenario'] == 'Baseline (Crisis)'].iloc[0]
hybrid   = scenarios[scenarios['Scenario'] == 'Hybrid (Optimized)'].iloc[0]

# ROI is taken from ROI_M — already net of intervention costs
sla_savings  = baseline['SLA_Penalty_M'] - hybrid['SLA_Penalty_M']
idle_savings = baseline['Idle_Cost_M']   - hybrid['Idle_Cost_M']
total_roi    = hybrid['ROI_M']  # $19.23M — the authoritative figure

reliability_delta  = hybrid['Reliability_Pct'] - baseline['Reliability_Pct']
vessels_recovered  = baseline['Vessels_Delayed'] - hybrid['Vessels_Delayed']
gemini_target      = 90.0

print('=== ECONOMIC IMPACT ===')
print(f'  Baseline total exposure:    '
      f'${baseline["SLA_Penalty_M"] + baseline["Idle_Cost_M"]:.2f}M')
print(f'  Optimized total exposure:   '
      f'${hybrid["SLA_Penalty_M"] + hybrid["Idle_Cost_M"]:.2f}M')
print()
print(f'  SLA penalty reduction:      '
      f'${sla_savings:.2f}M  ({sla_savings/baseline["SLA_Penalty_M"]:.0%})')
print(f'  Idle cost reduction:        '
      f'${idle_savings:.2f}M  ({idle_savings/baseline["Idle_Cost_M"]:.0%})')
print(f'  {"-"*44}')
print(f'  TOTAL Q1 VALUE RECOVERED:   ${total_roi:.2f}M')
print()
print('=== OPERATIONAL IMPROVEMENT ===')
print(f'  Schedule reliability:       '
      f'{baseline["Reliability_Pct"]}%  →  {hybrid["Reliability_Pct"]}%  '
      f'(+{reliability_delta:.1f}pp)')
print(f'  Vessels removed from risk:  '
      f'{vessels_recovered} of {int(baseline["Vessels_Delayed"])}')
print(f'  Gemini 90% target met:      '
      f'{"YES" if hybrid["Reliability_Pct"] >= gemini_target else "NO"}')
print(f'  Margin above target:        '
      f'+{hybrid["Reliability_Pct"] - gemini_target:.1f}pp')

## 5.0 Scenario Rankings

All four intervention scenarios are ranked by value recovered. The gap between single-lever approaches (Slow-Steam Only, Priority Berthing Only) and the Hybrid confirms that neither scheduling discipline alone is sufficient — the compounding effect of both is required to clear the Gemini reliability threshold.

In [ ]:
scenario_order = scenarios.sort_values('ROI_M', ascending=True)
bar_colors_s   = ['#D62828', '#FFD028', '#6dcee8', '#42B0D5']

fig, ax = plt.subplots(figsize=(10, 4))

bars = ax.barh(scenario_order['Scenario'],
               scenario_order['ROI_M'],
               color=bar_colors_s, edgecolor='none', height=0.5)

ax.axvline(x=0, color='#878787', linewidth=0.8)

for bar, val in zip(bars, scenario_order['ROI_M']):
    label = f'+${val}M' if val > 0 else 'Baseline'
    ax.text(val + 0.4, bar.get_y() + bar.get_height() / 2,
            label, va='center', color='#EDEDED', fontsize=9)

ax.set_title('Value Recovered vs. Crisis Baseline by Scenario',
             color='#EDEDED', fontsize=12, fontweight='bold', pad=14)
ax.set_xlabel('Value Created ($M)', color='#878787', fontsize=9)

for spine in ax.spines.values():
    spine.set_edgecolor('#1a4a6a')

plt.tight_layout()
plt.savefig('scenario_rankings.png', dpi=150,
            bbox_inches='tight', facecolor='#002147')
plt.show()

## 6.0 Strategic Implication

The Hybrid scenario outperforms every single-lever alternative by a factor of 2x or more. At 92.9% schedule reliability, it exceeds the Gemini Cooperation's 90% benchmark — positioning Rotterdam MVII as a reliable anchor in the alliance network during the stepwise return to Suez routing in early 2026.

The crisis window analysis confirms that Weeks 5–9 are not a random variance event. They are a structural consequence of route convergence that will recur whenever Suez access is disrupted. The scheduling interventions modelled here are therefore repeatable and deployable across any APM Terminals hub in the Gemini network.

At 60 terminals globally, the Hybrid scenario's efficiency profile represents an indicative **$1.14B in annual operational value** — before the 2027 Rotterdam expansion adds a single TEU of new capacity.